# Baseline BioBERTpt — RECLin-PT (Colab T4)

Treina o baseline de extração de relação em 3 classes (`negation_of`, `associated_with`, `no_relation`) sobre os splits do SemClinBr usando o **BioBERTpt** (`pucpr/biobertpt-all`), encoder **clínico (pré-treinado em textos clínicos)** do português.

É o **par de comparação** do baseline BERTimbau (encoder **geral**). Mesma representação de entrada, loss, scheduler, early stopping, salvamento, métricas, seed e logging — a **única** diferença é o checkpoint de pré-treino. Assim respondemos, de forma válida:

> **"O pré-treinamento clínico importa para extração de relações em textos médicos em português?"**

Métricas: Macro/Micro/Weighted-F1, **MCC**, F1 por classe, matriz de confusão e curvas de treino/validação. O **teste de significância** entre os dois modelos está no notebook `significance_colab.ipynb` (rode depois de treinar os dois).

Antes de rodar: **Ambiente de execução → Alterar o tipo de ambiente → GPU (T4)** e cadastre o secret `GITHUB_PAT` (ícone de chave 🔑).

O treino salva **`best_model/` + `last_checkpoint/` por época no Google Drive** e **retoma automaticamente** se o runtime cair.

## 1. Clonar o repositório

In [ ]:
import os, subprocess
from google.colab import userdata

TOKEN = userdata.get('GITHUB_PAT')           # secret do Colab, nunca hard-coded
REPO_NAME = 'RECLin-PT-Min'                   # nome do repo no GitHub
REPO = f'/content/{REPO_NAME}'
GIT_USER_NAME = 'angeloalsf'
GIT_USER_EMAIL = 'angeloalsf@gmail.com'

url = f'https://{TOKEN}@github.com/angeloalsf/{REPO_NAME}'
if not os.path.isdir(REPO):
    subprocess.run(['git', 'clone', url, REPO], check=True)
else:
    print('Repo já clonado; fazendo pull --rebase para integrar mudanças remotas.')
    subprocess.run(['git', '-C', REPO, 'pull', '--rebase', '--autostash'], check=False)

subprocess.run(['git', '-C', REPO, 'config', 'user.name', GIT_USER_NAME], check=True)
subprocess.run(['git', '-C', REPO, 'config', 'user.email', GIT_USER_EMAIL], check=True)
subprocess.run(['git', '-C', REPO, 'remote', 'set-url', 'origin', url], check=True)
print('Repo pronto em', REPO)


## 2. Verificar a GPU (T4)

In [ ]:
!nvidia-smi
import torch
print('CUDA disponível:', torch.cuda.is_available())
print('GPU:', torch.cuda.get_device_name(0) if torch.cuda.is_available() else '—')
assert torch.cuda.is_available(), 'Sem GPU! Ative em Ambiente de execução → GPU (T4).'


## 3. Montar o Google Drive (checkpoints)

Checkpoints em `MyDrive/RECLin-PT-Min/checkpoints_biobertpt/` — **pasta própria** deste modelo. Dentro: `best_model/` (pesos do melhor epoch) e `last_checkpoint/` (estado de retomada).

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

CKPT_DIR = f'/content/drive/MyDrive/{REPO_NAME}/checkpoints_biobertpt'
os.makedirs(CKPT_DIR, exist_ok=True)
print('Checkpoints em:', CKPT_DIR)
existe = os.path.isfile(os.path.join(CKPT_DIR, 'last_checkpoint', 'training_state.pt'))
print('last_checkpoint existente?', existe, '(se True, o treino vai RETOMAR de onde parou)')


## 4. Instalar dependências

O Colab já traz `torch` com CUDA — **não reinstalamos**.

In [ ]:
!pip install -q "transformers>=4.40" "scikit-learn>=1.4" lxml


## 5. Conferir os splits

Os splits (`data/splits/*.jsonl`) são versionados no repo (seed 42, nível de documento) — **os mesmos para os dois modelos**.

In [ ]:
import subprocess, sys
splits = os.path.join(REPO, 'data', 'splits')
need = ['train.jsonl', 'dev.jsonl', 'test.jsonl']
have = all(os.path.isfile(os.path.join(splits, f)) for f in need)

if not have:
    xml = os.path.join(REPO, 'SemClinBr-xml-public-v1')
    if os.path.isdir(xml) and os.listdir(xml):
        print('Splits ausentes; gerando a partir dos XMLs...')
        subprocess.run([sys.executable, 'src/parse_semclinbr.py',
                        '--xml-dir', 'SemClinBr-xml-public-v1',
                        '--out', 'data/processed/dataset.jsonl'], cwd=REPO, check=True)
        subprocess.run([sys.executable, 'src/make_splits.py'], cwd=REPO, check=True)
    else:
        raise FileNotFoundError('Splits não encontrados e XMLs ausentes.')

for f in need:
    n = sum(1 for _ in open(os.path.join(splits, f), encoding='utf-8'))
    print(f'{f}: {n} docs')


## 6. Treinar o baseline BioBERTpt (com retomada)

Encoder `pucpr/biobertpt-all`, marcadores `[E1] [/E1] [E2] [/E2]`, 3 classes, `class_weight=balanced`. Melhor época pelo macro-F1 no **dev**; **test** uma única vez.

**Hiperparâmetros idênticos entre os dois modelos.** Se o runtime cair, **rode esta célula de novo**: ele pula as épocas já feitas.

Multi-seed manual: mude `--seed`, e use `--out results/baseline_biobertpt_seed43.json` e um `--ckpt-dir` com sufixo `_seed43`.

In [ ]:
%cd $REPO
!python -u src/baseline_biobertpt.py \
    --splits-dir data/splits \
    --model pucpr/biobertpt-all \
    --epochs 3 \
    --batch-size 64 \
    --max-gap 20 \
    --max-length 128 \
    --seed 42 \
    --ckpt-dir "$CKPT_DIR" \
    --out results/baseline_biobertpt.json


## 7. Resultados (métricas + matriz de confusão)

In [ ]:
import json
r = json.load(open(os.path.join(REPO, 'results', 'baseline_biobertpt.json'), encoding='utf-8'))
print('Modelo:', r['model'], '| parâmetros:', f"{r['n_params']/1e6:.1f}M")
print('Candidatos:', r['n_candidates'])
print('Macro-F1 :', round(r['test_macro_f1'], 4))
print('Micro-F1 :', round(r['test_micro_f1'], 4))
print('Weighted :', round(r['test_weighted_f1'], 4))
print('MCC      :', round(r['test_mcc'], 4))
print('\nF1 por classe:')
for k, v in r['test_f1_per_class'].items():
    mark = '   <<< NEGAÇÃO' if k == 'negation_of' else ''
    print(f'  {k:<18s} {v:.4f}{mark}')

cm = r['confusion_matrix']; labels = cm['labels']
print('\nMatriz de confusão (linhas=verdadeiro, colunas=predito):')
print('true\\pred'.ljust(12) + ''.join(s[:8].rjust(10) for s in labels))
for i, l in enumerate(labels):
    print(l[:11].ljust(12) + ''.join(str(int(c)).rjust(10) for c in cm['matrix'][i]))


## 8. Curvas de treino/validação

De `dev_history`: loss de treino vs. validação (overfitting) e F1 no dev por época (seleção de época).

In [ ]:
import matplotlib.pyplot as plt
h = r['dev_history']
ep = [e['epoch'] for e in h]
fig, ax = plt.subplots(1, 2, figsize=(11, 4))
ax[0].plot(ep, [e['train_loss'] for e in h], 'o-', label='train_loss')
ax[0].plot(ep, [e.get('dev_loss') for e in h], 's-', label='dev_loss')
ax[0].set_xlabel('época'); ax[0].set_ylabel('loss'); ax[0].set_title('Loss'); ax[0].legend(); ax[0].grid(alpha=.3)
ax[1].plot(ep, [e['dev_macro_f1'] for e in h], 'o-', label='dev_macro_f1')
ax[1].plot(ep, [e['dev_negation_of_f1'] for e in h], 's-', label='dev_negation_of_f1')
ax[1].set_xlabel('época'); ax[1].set_ylabel('F1'); ax[1].set_title('F1 no dev'); ax[1].legend(); ax[1].grid(alpha=.3)
plt.tight_layout(); plt.show()


## 9. Salvar resultados + predições no GitHub

`results/` é gitignored; forçamos o `add` do JSON de métricas **e** do `.preds.json` (predições do test, usadas no notebook de significância). Ambos são pequenos.

In [ ]:
import subprocess
for f in ['results/baseline_biobertpt.json', 'results/baseline_biobertpt.preds.json']:
    subprocess.run(['git', '-C', REPO, 'add', '-f', f], check=True)
subprocess.run(['git', '-C', REPO, 'commit', '-m',
                'resultados: baseline BioBERTpt (Colab T4)'], check=False)
subprocess.run(['git', '-C', REPO, 'pull', '--rebase', '--autostash'], check=False)
subprocess.run(['git', '-C', REPO, 'push'], check=False)
print('Push concluído (se houve mudanças).')


## 10. (Opcional) Enviar o `best_model/` para o Hugging Face Hub

O modelo final (~440 MB) **não cabe no GitHub** (limite de 100 MB/arquivo). Para tirá-lo do Drive, envie ao **HF Hub** (grátis). Cadastre o secret `HF_TOKEN` e ajuste `HF_REPO`. Pule se não precisar.

In [ ]:
from google.colab import userdata
from huggingface_hub import HfApi

HF_REPO = 'seu-usuario/reclin-pt-biobertpt'   # ajuste para o seu namespace
best = os.path.join(CKPT_DIR, 'best_model')

api = HfApi(token=userdata.get('HF_TOKEN'))
api.create_repo(HF_REPO, repo_type='model', exist_ok=True)
api.upload_folder(folder_path=best, repo_id=HF_REPO, repo_type='model')
print('best_model enviado para https://huggingface.co/' + HF_REPO)
